<a href="https://colab.research.google.com/github/altardefey/simulador-de-impacto-orcamentario/blob/main/Simulador_Impacto_Orcamentario.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simulador de Impacto Orçamentário | Salvador – BA

Execute as células em ordem no Google Colab. Autorize o acesso ao Google Drive. Se a pasta Dados_turismo estiver ausente, envie o ZIP quando solicitado. Os valores gerados são estimativas condicionadas às entradas do cenário. A diferença de ISS exibida é uma comparação com o histórico, sem interpretação causal.

In [2]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile

import pandas as pd

pasta_dados = Path('/content/drive/MyDrive/Dados_turismo')
if not pasta_dados.is_dir():
    from google.colab import drive

    drive.mount('/content/drive')

colunas_por_base = [
    {'taxa_ocupacao'},
    {'diaria_media', 'revpar'},
    {'fator_evento'},
    {'receita_turistica'},
    {'receita_iss'},
]

def ler_csvs():
    if pasta_dados.is_dir():
        for arquivo in sorted(pasta_dados.glob('*.csv')):
            yield pd.read_csv(arquivo, encoding='utf-8-sig')
        return

    from google.colab import files

    print(f'Pasta ausente: {pasta_dados}. Envie o arquivo big data.zip.')
    uploads = files.upload()
    zips = [conteudo for nome, conteudo in uploads.items() if nome.lower().endswith('.zip')]
    if len(zips) != 1:
        raise ValueError('Envie um unico ZIP com os cinco CSVs.')
    with ZipFile(BytesIO(zips[0])) as arquivo_zip:
        for item in arquivo_zip.infolist():
            if item.filename.lower().endswith('.csv'):
                with arquivo_zip.open(item) as csv:
                    yield pd.read_csv(csv, encoding='utf-8-sig')

bases = {}
for tabela in ler_csvs():
    for colunas in colunas_por_base:
        if colunas <= set(tabela.columns):
            chave = frozenset(colunas)
            if chave in bases:
                raise ValueError(f'Mais de um CSV para {sorted(colunas)}.')
            bases[chave] = tabela[['ano_mes', *sorted(colunas)]].set_index('ano_mes', verify_integrity=True)

faltantes = [sorted(c) for c in colunas_por_base if frozenset(c) not in bases]
if faltantes:
    raise ValueError(f'Bases ausentes: {faltantes}')

meses = next(iter(bases.values())).index
for tabela in bases.values():
    if set(tabela.index) != set(meses):
        raise ValueError(f'Cobertura mensal diferente na base {list(tabela.columns)}.')

df_final = pd.concat(bases.values(), axis=1).sort_index()
df_final.index = pd.PeriodIndex(df_final.index, freq='M', name='ano_mes')
if not df_final.index.equals(pd.period_range(df_final.index.min(), df_final.index.max(), freq='M')):
    raise ValueError('Há lacunas ou duplicatas na sequência mensal.')
if df_final.isna().any().any():
    raise ValueError('Há valores ausentes na tabela consolidada.')
df_final['mes'] = df_final.index.month
print(f'{len(df_final)} meses consolidados: {df_final.index.min()} a {df_final.index.max()}.')
display(df_final.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pasta ausente: /content/drive/MyDrive/Dados_turismo. Envie o arquivo big data.zip.


Saving big data.zip to big data.zip
68 meses consolidados: 2021-01 a 2026-08.


,receita_turistica,receita_iss,taxa_ocupacao,diaria_media,revpar,fator_evento,mes
ano_mes,,,,,,,
2021-01,861629655.0,80272069.56,46.32,302.71,171.33,2,1
2021-02,709653958.0,86732424.40,38.15,291.01,133.98,2,2
2021-03,332597976.0,92669109.14,17.88,268.67,63.35,0,3
2021-04,373893697.0,87733324.20,20.10,264.05,62.32,0,4
2021-05,486433840.0,87708104.97,26.15,253.39,82.57,0,5


## Treino e validação

O RevPAR informado nos CSVs diverge da fórmula do painel em parte do histórico. Para usar a mesma definição no treino e na simulação, o modelo recebe o RevPAR calculado por ocupação × diária, com ocupação convertida de percentual para fração. O RevPAR original continua na tabela para comparação.

Os 12 meses mais recentes ficam reservados para teste. A referência simples repete o valor observado no mesmo mês do ano anterior. O MAE mede o erro absoluto médio em reais e o R² pode ser negativo. Os modelos são testados com ocupação e diária observadas, portanto as métricas avaliam estimativas condicionadas a essas entradas.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

def moeda(valor):
    numero = f'{abs(valor):,.2f}'.replace(',', '_').replace('.', ',').replace('_', '.')
    sinal = '-' if valor < 0 else ''
    return f'{sinal}R$ {numero}'

def calcular_revpar(ocupacao, diaria):
    return ocupacao / 100 * diaria

df_final['revpar_calculado'] = calcular_revpar(df_final['taxa_ocupacao'], df_final['diaria_media'])
diferenca = (df_final['revpar'] - df_final['revpar_calculado']).abs()
print(f'RevPAR informado difere do calculado em mais de R$ 0,05 em {(diferenca > 0.05).sum()} meses.')

entradas = ['taxa_ocupacao', 'diaria_media', 'revpar_calculado', 'fator_evento', 'mes']
alvos = {'receita_turistica': 'Receita turística', 'receita_iss': 'ISS'}
if len(df_final) < 24:
    raise ValueError('São necessários ao menos 24 meses para este corte de validação.')

treino = df_final.iloc[:-12]
teste = df_final.iloc[-12:]
print(f'Teste: {teste.index.min()} a {teste.index.max()}.')
modelos = {}
resultados = []
for coluna, nome in alvos.items():
    modelo = RandomForestRegressor(n_estimators=300, random_state=42)
    modelo.fit(treino[entradas], treino[coluna])
    observado = teste[coluna]
    estimado = modelo.predict(teste[entradas])
    referencia = treino.loc[teste.index - 12, coluna].to_numpy()
    for metodo, valores in [('Random Forest', estimado), ('Mesmo mês do ano anterior', referencia)]:
        resultados.append({
            'Alvo': nome,
            'Método': metodo,
            'MAE (R$)': mean_absolute_error(observado, valores),
            'R²': r2_score(observado, valores),
        })
    modelos[coluna] = modelo.fit(df_final[entradas], df_final[coluna])

metricas = pd.DataFrame(resultados).round({'R²': 3})
metricas['MAE (R$)'] = metricas['MAE (R$)'].map(moeda)
display(metricas)


## Painel de cenários

A referência é a média histórica observada para o mês selecionado. O painel usa os modelos ajustados com todo o histórico após a avaliação. As faixas dos controles cobrem o intervalo histórico de ocupação e diária. Combinações dentro dessas faixas ainda podem ser pouco representadas nos dados.

In [5]:
import math
import ipywidgets as widgets
from IPython.display import HTML, display

def simular(ocupacao, diaria, evento, mes):
    revpar = calcular_revpar(ocupacao, diaria)
    cenario = pd.DataFrame([[ocupacao, diaria, revpar, evento, mes]], columns=entradas)
    receita = modelos['receita_turistica'].predict(cenario)[0]
    iss = modelos['receita_iss'].predict(cenario)[0]
    historico = df_final.loc[df_final['mes'] == mes, list(alvos)].mean()
    receita_ref = historico['receita_turistica']
    iss_ref = historico['receita_iss']
    variacao = f'{receita / receita_ref - 1:+.1%}' if receita_ref else 'indisponível'

    display(HTML(
        f'<b>RevPAR do cenário:</b> {moeda(revpar)}<br>'
        f'<b>Receita turística estimada:</b> {moeda(receita)} '
        f'({variacao} ante a média do mês)<br>'
        f'<b>ISS estimado:</b> {moeda(iss)}<br>'
        f'<b>Diferença de ISS:</b> {moeda(iss - iss_ref)} '
        f'ante a média histórica do mês de {moeda(iss_ref)}'
    ))

def controle(coluna, titulo, passo):
    valores = df_final[coluna]
    return widgets.FloatSlider(
        value=float(valores.median()), min=math.floor(valores.min()),
        max=math.ceil(valores.max()), step=passo, description=titulo,
        continuous_update=False, style={'description_width': 'initial'},
    )

widgets.interact(
    simular,
    ocupacao=controle('taxa_ocupacao', 'Ocupação (%)', 0.1),
    diaria=controle('diaria_media', 'Diária (R$)', 1),
    evento=widgets.Dropdown(options=[('Regular', 0), ('Festival', 1), ('Alta temporada', 2)], description='Evento'),
    mes=widgets.Dropdown(options=range(1, 13), description='Mês'),
);

interactive(children=(FloatSlider(value=62.769999999999996, continuous_update=False, description='Ocupação (%)…